# API Smoke Test

Notebook này chỉ kiểm tra 3 việc:

1. `teacher-llm-test`: Teacher LLM nhận một bài báo ngắn và gán nhãn thử.
2. `student-llm-test`: Student LLM 8B nhận prompt ngắn và trả lời thử.
3. `embedding-test`: Embedding model nhận text và trả vector thử.

Notebook đọc cấu hình từ `.env` và không in API key.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path
from typing import Any

import requests


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "finevent").exists():
            return candidate
    raise RuntimeError("Không tìm thấy project root.")


PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)


def load_env(path: Path = PROJECT_ROOT / ".env") -> None:
    try:
        from dotenv import load_dotenv
    except ImportError:
        if not path.exists():
            return
        for raw_line in path.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))
    else:
        load_dotenv(path)


def require_env(name: str) -> str:
    value = os.getenv(name)
    if not value:
        raise RuntimeError(f"Thiếu biến môi trường: {name}")
    return value


def redact(value: str | None) -> str | None:
    if value is None:
        return None
    return "***" if len(value) <= 8 else f"{value[:4]}...{value[-4:]}"


def chat_completion(
    *,
    base_url: str,
    api_key: str,
    model: str,
    prompt: str,
    max_tokens: int = 512,
) -> dict[str, Any]:
    url = f"{base_url.rstrip('/')}/chat/completions"
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0,
        "max_tokens": max_tokens,
    }
    response = requests.post(
        url,
        headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
        json=payload,
        timeout=120,
    )
    if response.status_code == 400:
        payload["max_completion_tokens"] = payload.pop("max_tokens")
        response = requests.post(
            url,
            headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
            json=payload,
            timeout=120,
        )
    response.raise_for_status()
    return response.json()


def message_text(payload: dict[str, Any]) -> str:
    choices = payload.get("choices") or []
    if not choices:
        return ""
    message = choices[0].get("message") or {}
    content = message.get("content") or ""
    reasoning = message.get("reasoning_content") or ""
    return str(content or reasoning).strip()


def embedding_request(*, base_url: str, api_key: str, model: str, texts: list[str]) -> dict[str, Any]:
    response = requests.post(
        f"{base_url.rstrip('/')}/embeddings",
        headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
        json={"model": model, "input": texts},
        timeout=120,
    )
    response.raise_for_status()
    return response.json()


load_env()
print(json.dumps({"project_root": str(PROJECT_ROOT), "env_file_exists": (PROJECT_ROOT / ".env").exists()}, ensure_ascii=False, indent=2))

In [ ]:
teacher_base_url = os.getenv("TEACHER_LLM_BASE_URL") or "https://api.openai.com/v1"
teacher_api_key = require_env("OPENAI_API_KEY")
teacher_model = require_env("TEACHER_LLM_MODEL")

teacher_prompt = """
Bạn là mô hình gán nhãn sự kiện tài chính tiếng Việt. Trả về JSON ngắn.
Bài báo: HPG công bố khởi công nhà máy mới tại Bình Dương, dự kiến tăng công suất sản xuất từ năm sau.
Schema: {"document_label":"HAS_EVENT|NO_EVENT","events":[{"event_type":"EXPANSION","impact_sentiment":"POSITIVE|NEGATIVE|NEUTRAL","evidence_span":"..."}]}
""".strip()

teacher_payload = chat_completion(
    base_url=teacher_base_url,
    api_key=teacher_api_key,
    model=teacher_model,
    prompt=teacher_prompt,
    max_tokens=512,
)
teacher_text = message_text(teacher_payload)
print(json.dumps({"test": "teacher-llm-test", "model": teacher_model, "api_key": redact(teacher_api_key), "content_preview": teacher_text[:1000]}, ensure_ascii=False, indent=2))

In [ ]:
student_base_url = require_env("STUDENT_LLM_BASE_URL")
student_api_key = require_env("STUDENT_LLM_API_KEY")
student_model = require_env("STUDENT_LLM_MODEL")

student_prompt = "/no_think\nTrả lời JSON ngắn: {\"status\":\"ok\", \"task\":\"student smoke test\"}."
student_payload = chat_completion(
    base_url=student_base_url,
    api_key=student_api_key,
    model=student_model,
    prompt=student_prompt,
    max_tokens=512,
)
student_text = message_text(student_payload)
print(json.dumps({"test": "student-llm-test", "model": student_model, "api_key": redact(student_api_key), "content_preview": student_text[:1000]}, ensure_ascii=False, indent=2))

In [ ]:
embedding_base_url = require_env("EMBEDDING_BASE_URL")
embedding_api_key = require_env("EMBEDDING_API_KEY")
embedding_model = require_env("EMBEDDING_MODEL")

embedding_payload = embedding_request(
    base_url=embedding_base_url,
    api_key=embedding_api_key,
    model=embedding_model,
    texts=["HPG khởi công nhà máy mới và mở rộng năng lực sản xuất."],
)
vector = embedding_payload["data"][0]["embedding"]
print(json.dumps({"test": "embedding-test", "model": embedding_model, "api_key": redact(embedding_api_key), "dimension": len(vector), "first_5": vector[:5]}, ensure_ascii=False, indent=2))